## Notebook Data Analysis para o RCE Framework Laucher usando Função objetivo - IEEE 14
---

## 1) RCE Datasset

In [12]:
import pandas as pd


df_mutacao_100_gen = pd.read_excel("/home/pedrov12/Documentos/GitHub/Repopulation-With-Elite-Set/resultados - Artigo PIBIC/dataset/resultados_consolidados_variando_mutaca_4x.xlsx")

df_resultados_4x_mutacao = pd.read_excel("/home/pedrov12/Documentos/GitHub/Repopulation-With-Elite-Set/resultados - Artigo PIBIC/dataset/resultados_consolidados_variando_mutaca_4x.xlsx")

df_resultados_Agosto_Laucher_Streamlit = pd.read_csv("/home/pedrov12/Documentos/GitHub/Repopulation-With-Elite-Set/resultados - Artigo PIBIC/dataset/resultados_consolidados_Agosto_Laucher_Streamlit.csv")



#### Este DF foi gerado pelo Launcher em output/resultados_consolidados.xlsx

In [13]:


df_resultados_Agosto_Laucher_Streamlit.head(3)

,pasta_run,configuracao,execucao,config_num,param_CROSSOVER,param_DELTA_MIN,param_IND_SIZE,param_MUTACAO,param_NUM_GENERATIONS,param_NUM_VAR_DIFERENTES,param_POP_SIZE,param_PORCENTAGEM,param_RCE_REPOPULATION_GENERATIONS,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,best_fitness,best_gen_idx
0,run_2025-08-21_22-11-53,config_1,1,1,0.9,2,5,0.9,5,1,5,0.2,50,16,17,16,1,27,54.452415,4
1,run_2025-08-21_22-11-53,config_1,2,1,0.9,2,5,0.9,5,1,5,0.2,50,23,31,10,10,3,61.097299,4
2,run_2025-08-21_22-11-53,config_1,3,1,0.9,2,5,0.9,5,1,5,0.2,50,0,23,14,29,2,63.263155,4


In [14]:

best_variables_df = df_resultados_Agosto_Laucher_Streamlit[["best_fitness","best_var_1", "best_var_2", "best_var_3", "best_var_4", "best_var_5","configuracao", "execucao",]]

for i in range(best_variables_df.index.start + 1,best_variables_df.index.stop):
    pass
display(best_variables_df)
(best_variables_df.groupby("configuracao")).mean()


,best_fitness,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,configuracao,execucao
0,54.452415,16,17,16,1,27,config_1,1
1,61.097299,23,31,10,10,3,config_1,2
2,63.263155,0,23,14,29,2,config_1,3
3,53.891490,10,30,24,10,18,config_1,4
4,60.997438,7,6,22,22,7,config_1,5
...,...,...,...,...,...,...,...,...
512,56.631527,10,20,4,10,4,config_2,1
513,53.785858,13,2,22,13,13,config_2,2
514,38.553758,26,16,26,26,17,config_2,3
515,73.086615,31,29,29,18,31,config_2,4


,best_fitness,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,execucao
configuracao,,,,,,,
config_1,47.138218,16.427184,14.849515,15.645631,15.509709,17.004854,3.213592
config_2,46.412362,15.376712,15.561644,15.589041,16.164384,16.431507,2.958904
config_3,41.928687,17.529412,13.164706,17.364706,17.541176,16.776471,3.000000
config_4,40.002528,17.475000,12.800000,17.112500,16.937500,17.325000,3.000000


## 2) Limpeza e Tratamento dos resultados consolidados

In [15]:
# --- Fluxo Corrigido ---

# Primeiro, precisamos da função que cria o molde
def renomear_colunas_from_map(df):
  mapa_inicial = {coluna: coluna for coluna in df.columns}
  return mapa_inicial

# 1. Crie o "molde" usando a função correta
mapa_molde = renomear_colunas_from_map(df_resultados_Agosto_Laucher_Streamlit)

# 2. Edite o molde como você já fez (esta parte está perfeita!)
mapa_molde['param_CROSSOVER'] = 'Crossover'
mapa_molde['param_MUTACAO'] = 'Mutação'
mapa_molde['param_NUM_GENERATIONS'] = 'Num_Gerações'
mapa_molde["param_IND_SIZE"] = 'tamanho_individuo'
# ...

# 3. Use o mapa já modificado para renomear, pode ser com o .rename() simples
results_df = df_resultados_Agosto_Laucher_Streamlit.rename(columns=mapa_molde)

# 4. Verifique o resultado
print("\nColunas renomeadas com sucesso!")
results_df.head(3)


Colunas renomeadas com sucesso!


,pasta_run,configuracao,execucao,config_num,Crossover,param_DELTA_MIN,tamanho_individuo,Mutação,Num_Gerações,param_NUM_VAR_DIFERENTES,param_POP_SIZE,param_PORCENTAGEM,param_RCE_REPOPULATION_GENERATIONS,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,best_fitness,best_gen_idx
0,run_2025-08-21_22-11-53,config_1,1,1,0.9,2,5,0.9,5,1,5,0.2,50,16,17,16,1,27,54.452415,4
1,run_2025-08-21_22-11-53,config_1,2,1,0.9,2,5,0.9,5,1,5,0.2,50,23,31,10,10,3,61.097299,4
2,run_2025-08-21_22-11-53,config_1,3,1,0.9,2,5,0.9,5,1,5,0.2,50,0,23,14,29,2,63.263155,4


In [16]:
def aplicar_renomeacao(df, mapa_de_nomes, manter_apenas_mapeadas=False):
  """
  Renomeia colunas de um DataFrame e, opcionalmente, mantém apenas as
  colunas que foram especificadas no mapa de renomeação.

  Args:
    df (pd.DataFrame): O DataFrame original.
    mapa_de_nomes (dict): Dicionário com {nome_antigo: nome_novo}.
    manter_apenas_mapeadas (bool): Se True, descarta todas as outras colunas.
  
  Returns:
    pd.DataFrame: O DataFrame transformado.
  """
  # Primeiro, renomeia as colunas normalmente
  df_renomeado = df.rename(columns=mapa_de_nomes)

  # Agora, a nova lógica que você sugeriu:
  if manter_apenas_mapeadas:
    # Pega a lista dos novos nomes de coluna (os valores do dicionário)
    colunas_para_manter = list(mapa_de_nomes.values())
    
    # Retorna o DataFrame contendo APENAS essas colunas
    return df_renomeado[colunas_para_manter]
  else:
    # Se for False, retorna o DataFrame com todas as colunas (com os nomes trocados)
    return df_renomeado

# --- Como usar ---

# 1. Crie um mapa APENAS com as colunas que te interessam
mapa_essencial = {
      "pasta_run":"pasta_run",
    'config_num':"Número Config",
    'param_CROSSOVER': 'Crossover', # Pode mapear para o mesmo nome
    'param_MUTACAO': 'Mutação',
    'param_NUM_GENERATIONS': 'Número de Gerações',
    'param_POP_SIZE': "Tamanho População",
    "param_IND_SIZE": "Tamanho Individuo",
    "best_gen_idx": "Melhor Geração",
    
}


# Use o DataFrame que já tinha alguns nomes trocados, o df_novo
# Caso de uso 1: Apenas renomear 'melhor_fitness' para 'fitness_final'
df_alterado = aplicar_renomeacao(df_resultados_Agosto_Laucher_Streamlit, mapa_essencial,True)

print("--- Rodando com manter_apenas_mapeadas=True ---")
display(df_alterado.head())


print("Resultados gerais para cada configuração executada")
#df_resultado = pd.merge(df_alterado, best_variables_df, on='best_fitness')
df_resultado = pd.concat([df_alterado, best_variables_df], axis=1)
df_resultado

--- Rodando com manter_apenas_mapeadas=True ---


,pasta_run,Número Config,Crossover,Mutação,Número de Gerações,Tamanho População,Tamanho Individuo,Melhor Geração
0,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4
1,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4
2,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4
3,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4
4,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4


Resultados gerais para cada configuração executada


,pasta_run,Número Config,Crossover,Mutação,Número de Gerações,Tamanho População,Tamanho Individuo,Melhor Geração,best_fitness,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,configuracao,execucao
0,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4,54.452415,16,17,16,1,27,config_1,1
1,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4,61.097299,23,31,10,10,3,config_1,2
2,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4,63.263155,0,23,14,29,2,config_1,3
3,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4,53.891490,10,30,24,10,18,config_1,4
4,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4,60.997438,7,6,22,22,7,config_1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,run_2025-08-25_11-46-54,2,0.5,0.8,20,5,5,19,56.631527,10,20,4,10,4,config_2,1
513,run_2025-08-25_11-46-54,2,0.5,0.8,20,5,5,19,53.785858,13,2,22,13,13,config_2,2
514,run_2025-08-25_11-46-54,2,0.5,0.8,20,5,5,19,38.553758,26,16,26,26,17,config_2,3
515,run_2025-08-25_11-46-54,2,0.5,0.8,20,5,5,19,73.086615,31,29,29,18,31,config_2,4


## Tratamento para o tempo de execução

In [17]:
import pandas as pd

def validate_time_laucher(df):
    """
    Calcula o tempo médio de execução para cada configuração por dia.

    A função executa os seguintes passos:
    1. Converte a coluna 'pasta_run' para um formato de data/hora.
    2. Cria uma coluna 'data' para agrupar as execuções por dia.
    3. Agrupa os dados por 'data' e 'configuracao'.
    4. Calcula a duração total das execuções para cada grupo.
    5. Conta o número de execuções em cada grupo.
    6. Calcula o tempo médio de execução e o adiciona como uma nova coluna.

    Args:
      df (pd.DataFrame): O DataFrame original contendo a coluna 'pasta_run'.

    Returns:
      pd.DataFrame: O DataFrame com as novas colunas de tempo e data.
    """
    df_calculado = df.copy()

    # Passo 1: Limpar a string e converter para datetime
    timestamp_str = df_calculado['pasta_run'].str.replace('run_', '')
    df_calculado['timestamp'] = pd.to_datetime(timestamp_str, format='%Y-%m-%d_%H-%M-%S')

    # Passo 2: Extrair a data para uma nova coluna
    df_calculado['data'] = df_calculado['timestamp'].dt.date

    # Passo 3, 4 e 5: Agrupar e calcular duração e contagem
    grupos = ['data', 'configuracao']
    tempo_min_grupo = df_calculado.groupby(grupos)['timestamp'].transform('min')
    tempo_max_grupo = df_calculado.groupby(grupos)['timestamp'].transform('max')
    duracao_total = tempo_max_grupo - tempo_min_grupo
    num_execucoes = df_calculado.groupby(grupos)['execucao'].transform('count')

    # Passo 6: Calcular o tempo médio
    df_calculado['tempo_medio_execucao'] = duracao_total / num_execucoes

    return df_calculado
#na coluna hora vc tem que fazer todos os horarios do dia um hora pela outra para saber o tempo de execucao de cada config e dividir pelo número de execuções e uma coluna data para saber o dia

df_resultado[["pasta_run"]].value_counts()


df_time = validate_time_laucher(df_resultado)
df_time

,pasta_run,Número Config,Crossover,Mutação,Número de Gerações,Tamanho População,Tamanho Individuo,Melhor Geração,best_fitness,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,configuracao,execucao,timestamp,data,tempo_medio_execucao
0,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4,54.452415,16,17,16,1,27,config_1,1,2025-08-21 22:11:53,2025-08-21,0 days 00:02:01
1,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4,61.097299,23,31,10,10,3,config_1,2,2025-08-21 22:11:53,2025-08-21,0 days 00:02:01
2,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4,63.263155,0,23,14,29,2,config_1,3,2025-08-21 22:11:53,2025-08-21,0 days 00:02:01
3,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4,53.891490,10,30,24,10,18,config_1,4,2025-08-21 22:11:53,2025-08-21,0 days 00:02:01
4,run_2025-08-21_22-11-53,1,0.9,0.9,5,5,5,4,60.997438,7,6,22,22,7,config_1,5,2025-08-21 22:11:53,2025-08-21,0 days 00:02:01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,run_2025-08-25_11-46-54,2,0.5,0.8,20,5,5,19,56.631527,10,20,4,10,4,config_2,1,2025-08-25 11:46:54,2025-08-25,0 days 00:01:20.396226415
513,run_2025-08-25_11-46-54,2,0.5,0.8,20,5,5,19,53.785858,13,2,22,13,13,config_2,2,2025-08-25 11:46:54,2025-08-25,0 days 00:01:20.396226415
514,run_2025-08-25_11-46-54,2,0.5,0.8,20,5,5,19,38.553758,26,16,26,26,17,config_2,3,2025-08-25 11:46:54,2025-08-25,0 days 00:01:20.396226415
515,run_2025-08-25_11-46-54,2,0.5,0.8,20,5,5,19,73.086615,31,29,29,18,31,config_2,4,2025-08-25 11:46:54,2025-08-25,0 days 00:01:20.396226415


In [18]:
import pandas as pd

def calcula_tempo_execucao(df):
    """
    Calcula o tempo médio de execução para cada grupo de configuração
    e adiciona como uma nova coluna no DataFrame.
    """
    # É uma boa prática trabalhar com uma cópia para não modificar o df original inesperadamente
    df_calculado = df.copy()

    # 1. Converte a coluna 'pasta_run' para um formato de data e hora (timestamp)
    # Primeiro removemos o "run_" do início do texto
    df_calculado['timestamp'] = df_calculado['pasta_run'].str.replace('run_', '')
    # Agora convertemos o texto restante para data/hora
    df_calculado['timestamp'] = pd.to_datetime(df_calculado['timestamp'], format='%Y-%m-%d_%H-%M-%S')

    # 2. Para cada grupo 'configuracao', calcula a duração total
    # A função .transform() é ótima aqui, pois ela realiza o cálculo por grupo
    # mas retorna um resultado com o mesmo tamanho do DataFrame original.
    tempo_min_grupo = df_calculado.groupby('configuracao')['timestamp'].transform('min')
    tempo_max_grupo = df_calculado.groupby('configuracao')['timestamp'].transform('max')

    duracao_total = tempo_max_grupo - tempo_min_grupo

    # 3. Conta o número de execuções em cada grupo
    num_execucoes = df_calculado.groupby('configuracao')['execucao'].transform('count')

    # 4. Calcula o tempo médio e cria a nova coluna
    # Verificamos se num_execucoes > 0 para evitar divisão por zero
    if not num_execucoes.empty:
      df_calculado['tempo_medio_execucao'] = duracao_total / num_execucoes
    else:
      df_calculado['tempo_medio_execucao'] = pd.NaT # preenche com nulo se não houver execuções

    return df_calculado




In [19]:


# --- Como usar tudo junto ---

# 1. Renomeie o DataFrame original
df_novo = df_resultado.rename(columns=mapa_molde)

# 2. Passe o DataFrame renomeado para a sua nova função
df_final = calcula_tempo_execucao(df_novo)

# 3. Veja o resultado!
# Usamos o .head() para ver as primeiras linhas e as novas colunas
print("\nDataFrame final com a coluna de tempo médio:")
display(df_final[['configuracao', 'execucao','timestamp', 'tempo_medio_execucao']].head())


# Supondo que seu DataFrame se chame 'df_final' e a coluna 'tempo_medio_execucao'
formatar_tempo = lambda td: f"{int(td.total_seconds() // 3600):02d}:{int((td.total_seconds() % 3600) // 60):02d}:{int(td.total_seconds() % 60):02d}"

# 2. Aplicamos essa função na coluna e criamos uma nova coluna formatada
df_final['tempo_formatado'] = df_final['tempo_medio_execucao'].apply(formatar_tempo)

# 3. Vamos ver o resultado
print("Coluna original vs. Coluna formatada:")
display(df_final[['tempo_medio_execucao', 'tempo_formatado']].head())

# A lambda pega o tempo (td), calcula o total de segundos,
# e formata em minutos (:02d) e segundos (:02d)
formatar_min_sec = lambda td: f"{int(td.total_seconds() // 60):02d}:{int(td.total_seconds() % 60):02d}"


#df_final['tempo_min_sec'] = df_final['tempo_medio_execucao'].apply(formatar_min_sec)

#print("Coluna original vs. Coluna formatada (MM:SS):")
#display(df_final[['tempo_medio_execucao', 'tempo_min_sec']].head())


print("DF Resultados Final")
df_final = df_final.drop(
  columns=['pasta_run', 'tempo_medio_execucao', 'timestamp',"Número Config"]
)

print("Resultados Finais tratados para o Dashboard")
df_final.to_excel("/home/pedrov12/Documentos/GitHub/Repopulation-With-Elite-Set/src/output/resultados.xlsx", index=False)

df_final




DataFrame final com a coluna de tempo médio:


,configuracao,execucao,timestamp,tempo_medio_execucao
0,config_1,1,2025-08-21 22:11:53,0 days 00:24:55.830097087
1,config_1,2,2025-08-21 22:11:53,0 days 00:24:55.830097087
2,config_1,3,2025-08-21 22:11:53,0 days 00:24:55.830097087
3,config_1,4,2025-08-21 22:11:53,0 days 00:24:55.830097087
4,config_1,5,2025-08-21 22:11:53,0 days 00:24:55.830097087


Coluna original vs. Coluna formatada:


,tempo_medio_execucao,tempo_formatado
0,0 days 00:24:55.830097087,00:24:55
1,0 days 00:24:55.830097087,00:24:55
2,0 days 00:24:55.830097087,00:24:55
3,0 days 00:24:55.830097087,00:24:55
4,0 days 00:24:55.830097087,00:24:55


DF Resultados Final
Resultados Finais tratados para o Dashboard


,Crossover,Mutação,Número de Gerações,Tamanho População,Tamanho Individuo,Melhor Geração,best_fitness,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,configuracao,execucao,tempo_formatado
0,0.9,0.9,5,5,5,4,54.452415,16,17,16,1,27,config_1,1,00:24:55
1,0.9,0.9,5,5,5,4,61.097299,23,31,10,10,3,config_1,2,00:24:55
2,0.9,0.9,5,5,5,4,63.263155,0,23,14,29,2,config_1,3,00:24:55
3,0.9,0.9,5,5,5,4,53.891490,10,30,24,10,18,config_1,4,00:24:55
4,0.9,0.9,5,5,5,4,60.997438,7,6,22,22,7,config_1,5,00:24:55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,0.5,0.8,20,5,5,19,56.631527,10,20,4,10,4,config_2,1,00:35:10
513,0.5,0.8,20,5,5,19,53.785858,13,2,22,13,13,config_2,2,00:35:10
514,0.5,0.8,20,5,5,19,38.553758,26,16,26,26,17,config_2,3,00:35:10
515,0.5,0.8,20,5,5,19,73.086615,31,29,29,18,31,config_2,4,00:35:10


## 3) Criação de tabelas e fazer seus UML com Diagrama de Classes com o uso do software

In [20]:

# tabela RCE


# Tabela AG-DEAP


# Tabela_Solutions

In [21]:
def separar_em_tabelas(df, coluna_criterio):
  """
  Separa um DataFrame em um dicionário de DataFrames menores,
  baseado nos valores únicos de uma coluna.

  Args:
    df (pd.DataFrame): O DataFrame a ser separado.
    coluna_criterio (str): O nome da coluna usada para agrupar.

  Returns:
    dict: Um dicionário onde as chaves são os valores da coluna
          e os valores são os DataFrames correspondentes.
  """
  tabelas_separadas = {}
  
  # O .groupby() cria um objeto que podemos percorrer com um loop.
  # Em cada passo, ele nos dá o nome do grupo e o DataFrame daquele grupo.
  for nome_do_grupo, df_do_grupo in df.groupby(coluna_criterio):
    # Usamos o nome do grupo como a chave do nosso dicionário
    # e guardamos o DataFrame correspondente como o valor.
    tabelas_separadas[nome_do_grupo] = df_do_grupo.copy() # .copy() evita avisos do pandas
  
  return tabelas_separadas

# --- Como usar a função ---
NUMERO_TABELS = 5 
# Vamos usar seu DataFrame final e separá-lo por 'configuracao'
dicionario_de_tabelas = separar_em_tabelas(df_final, 'configuracao')

# Agora, 'dicionario_de_tabelas' contém todas as suas tabelas separadas.
print("Tabelas criadas:", list(dicionario_de_tabelas.keys()))

for i in range(1,NUMERO_TABELS):
  df_config_1 = dicionario_de_tabelas[f'config_{i}']
  print(f"\nCabeçalho da tabela apenas para 'config_{i}':")
  display(df_config_1)


Tabelas criadas: ['config_1', 'config_2', 'config_3', 'config_4']

Cabeçalho da tabela apenas para 'config_1':


,Crossover,Mutação,Número de Gerações,Tamanho População,Tamanho Individuo,Melhor Geração,best_fitness,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,configuracao,execucao,tempo_formatado
0,0.9,0.9,5,5,5,4,54.452415,16,17,16,1,27,config_1,1,00:24:55
1,0.9,0.9,5,5,5,4,61.097299,23,31,10,10,3,config_1,2,00:24:55
2,0.9,0.9,5,5,5,4,63.263155,0,23,14,29,2,config_1,3,00:24:55
3,0.9,0.9,5,5,5,4,53.891490,10,30,24,10,18,config_1,4,00:24:55
4,0.9,0.9,5,5,5,4,60.997438,7,6,22,22,7,config_1,5,00:24:55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507,0.5,0.9,20,5,5,19,52.923361,31,20,5,5,31,config_1,1,00:24:55
508,0.5,0.9,20,5,5,19,53.249402,14,4,26,26,16,config_1,2,00:24:55
509,0.5,0.9,20,5,5,19,53.023765,28,16,2,2,28,config_1,3,00:24:55
510,0.5,0.9,20,5,5,19,52.399192,29,31,25,29,25,config_1,4,00:24:55



Cabeçalho da tabela apenas para 'config_2':


,Crossover,Mutação,Número de Gerações,Tamanho População,Tamanho Individuo,Melhor Geração,best_fitness,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,configuracao,execucao,tempo_formatado
5,0.9,0.77,5,5,5,4,64.461031,17,0,10,25,6,config_2,1,00:35:10
6,0.9,0.77,5,5,5,4,51.651970,7,2,7,29,15,config_2,2,00:35:10
7,0.9,0.77,5,5,5,4,53.891490,4,10,25,4,17,config_2,3,00:35:10
8,0.9,0.77,5,5,5,4,50.924353,18,7,18,30,7,config_2,4,00:35:10
9,0.9,0.77,5,5,5,4,63.210309,16,1,23,30,6,config_2,5,00:35:10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,0.5,0.80,20,5,5,19,56.631527,10,20,4,10,4,config_2,1,00:35:10
513,0.5,0.80,20,5,5,19,53.785858,13,2,22,13,13,config_2,2,00:35:10
514,0.5,0.80,20,5,5,19,38.553758,26,16,26,26,17,config_2,3,00:35:10
515,0.5,0.80,20,5,5,19,73.086615,31,29,29,18,31,config_2,4,00:35:10



Cabeçalho da tabela apenas para 'config_3':


,Crossover,Mutação,Número de Gerações,Tamanho População,Tamanho Individuo,Melhor Geração,best_fitness,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,configuracao,execucao,tempo_formatado
24,0.9,0.5,100,5,5,99,50.869004,2,3,20,20,2,config_3,1,00:24:27
25,0.9,0.5,100,5,5,99,52.784193,20,2,16,20,16,config_3,2,00:24:27
26,0.9,0.5,100,5,5,99,52.327280,13,22,21,21,21,config_3,3,00:24:27
27,0.9,0.5,100,5,5,99,52.364338,9,11,5,9,5,config_3,4,00:24:27
28,0.9,0.5,100,5,5,99,38.636266,16,2,16,16,3,config_3,5,00:24:27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,0.7,0.9,100,10,5,99,37.354029,20,11,20,20,22,config_3,1,00:24:27
340,0.7,0.9,100,10,5,99,37.354029,27,5,27,27,29,config_3,2,00:24:27
341,0.7,0.9,100,10,5,99,37.354029,29,20,29,29,29,config_3,3,00:24:27
342,0.7,0.9,100,10,5,99,50.869004,2,2,17,17,4,config_3,4,00:24:27



Cabeçalho da tabela apenas para 'config_4':


,Crossover,Mutação,Número de Gerações,Tamanho População,Tamanho Individuo,Melhor Geração,best_fitness,best_var_1,best_var_2,best_var_3,best_var_4,best_var_5,configuracao,execucao,tempo_formatado
44,0.9,0.25,100,5,5,99,76.592843,24,28,12,2,24,config_4,1,00:25:39
45,0.9,0.25,100,5,5,99,52.368205,1,23,22,22,3,config_4,2,00:25:39
46,0.9,0.25,100,5,5,99,38.534702,1,8,1,1,8,config_4,3,00:25:39
47,0.9,0.25,100,5,5,99,52.327280,23,2,2,2,4,config_4,4,00:25:39
48,0.9,0.25,100,5,5,99,52.368205,29,20,19,19,29,config_4,5,00:25:39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
344,0.6,0.90,100,10,5,99,37.354029,23,6,23,23,25,config_4,1,00:25:39
345,0.6,0.90,100,10,5,99,37.354029,22,3,22,22,24,config_4,2,00:25:39
346,0.6,0.90,100,10,5,99,50.869004,26,26,17,17,27,config_4,3,00:25:39
347,0.6,0.90,100,10,5,99,37.354029,14,21,14,14,14,config_4,4,00:25:39


## 3) Análisando os parametros AG

In [22]:
def display_value_counts(df, column_name):
    return df[column_name].value_counts()


print("Total de execuções pelo Laucher.py", df_final.shape[0])
print("Colunas do DataFrame",df_final.columns)

display(display_value_counts(df_final, 'Crossover'))
display(display_value_counts(df_final, 'Mutação'))
display(display_value_counts(df_final, 'Número de Gerações'))
display(display_value_counts(df_final, 'Tamanho População'))
display(display_value_counts(df_final, 'Melhor Geração'))




Total de execuções pelo Laucher.py 517
Colunas do DataFrame Index(['Crossover', 'Mutação', 'Número de Gerações', 'Tamanho População',
       'Tamanho Individuo', 'Melhor Geração', 'best_fitness', 'best_var_1',
       'best_var_2', 'best_var_3', 'best_var_4', 'best_var_5', 'configuracao',
       'execucao', 'tempo_formatado'],
      dtype='object')


Crossover
0.9    129
0.5    108
0.8     78
0.7     75
0.6     75
0.1     52
Name: count, dtype: int64

Mutação
0.90     420
0.80      53
0.77      19
90.00     10
0.50      10
0.25       5
Name: count, dtype: int64

Número de Gerações
100    347
20     160
5       10
Name: count, dtype: int64

Tamanho População
10    360
5     157
Name: count, dtype: int64

Melhor Geração
99    347
19    160
4      10
Name: count, dtype: int64

## 4) Análisando os parametros RCE

## 4 fitness functions

## Config params.json to 20x executions

## Simulate Pandapower network